# `model_experiment_NBEATS.ipynb`
## Walmart Recruiting - Store Sales Forecasting — N-BEATS (Deep Learning)

**არქიტექტურა:** N-BEATS (Neural Basis Expansion Analysis for Time Series) — `neuralforecast` (Nixtla) იმპლემენტაციით, გლობალური მოდელი, რომელიც ერთდროულად ვარჯიშობს ყველა Store-Dept წყვილის დროით მწკრივზე.

**ამ notebook-ის სტრუქტურა:**
1. Setup & MLflow tracking
2. Data Cleaning
3. Feature Engineering
4. Feature Selection (hist/futr exogenous features NBEATSx-სთვის)
5. Train / Validation split (time-based holdout, ბოლო 39 კვირა)
6. Custom WMAE მეტრიკა (Kaggle-ის ოფიციალური შეფასების წესით — holiday კვირები x5 წონა)
7. Hyperparameter Search — თითოეული run შემოსაზღვრულია **max 25 წუთით** (`trainer` `max_time`)
8. საუკეთესო კონფიგურაციის საბოლოო ვარჯიში სრულ მონაცემებზე
9. Pipeline-ის შენახვა MLflow Model Registry-ში (`WalmartSales_NBEATS`)

> ყველა experiment იწერება MLflow-ში ექსპერიმენტის სახელით `NBEATS_Training`, run-ებით: `NBEATS_Cleaning`, `NBEATS_FeatureEngineering`, `NBEATS_FeatureSelection`, `NBEATS_HPSearch_*`, `NBEATS_Final_Training` — დავალების სტრუქტურის შესაბამისად.


## 1. Setup

In [ ]:
# დააყენეთ საჭირო ბიბლიოთეკები (Kaggle/Colab გარემოში პირველად გაუშვით)
!pip install -q neuralforecast==1.7.4 mlflow dagshub pandas numpy scikit-learn matplotlib seaborn


In [ ]:
import os
import time
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.pyfunc

from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
from neuralforecast.losses.pytorch import MAE

warnings.filterwarnings("ignore")
np.random.seed(42)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)


In [ ]:
# --- MLflow / DagsHub tracking setup ---
# გუნდის საერთო რეპოზიტორიის მიხედვით შეცვალეთ owner/repo-name
import dagshub

DAGSHUB_OWNER = "sophyrise"          # <- შეცვალეთ საჭიროებისამებრ
DAGSHUB_REPO  = "walmart-sales-forecasting"  # <- შეცვალეთ საჭიროებისამებრ

dagshub.init(repo_owner=DAGSHUB_OWNER, repo_name=DAGSHUB_REPO, mlflow=True)

EXPERIMENT_NAME = "NBEATS_Training"
mlflow.set_experiment(EXPERIMENT_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())


In [ ]:
# --- გლობალური კონფიგურაცია ---
DATA_DIR = "/kaggle/input/walmart-recruiting-store-sales-forecasting"  # საჭიროებისამებრ შეცვალეთ
HORIZON = 39            # test.csv-ის კვირების რაოდენობა (2012-11-02 -> 2013-07-26)
MAX_RUN_MINUTES = 25    # ყოველი ვარჯიშის run დროის ჭერი (< 30 წთ მოთხოვნა, ბუფერით)
HP_SEARCH_SAMPLE_N = 300   # hyperparameter search-ისთვის სემფლირებული Store-Dept სერიების რაოდენობა
RANDOM_SEED = 42


## 2. Data Cleaning

In [ ]:
train_raw = pd.read_csv(f"{DATA_DIR}/train.csv", parse_dates=["Date"])
test_raw = pd.read_csv(f"{DATA_DIR}/test.csv", parse_dates=["Date"])
stores = pd.read_csv(f"{DATA_DIR}/stores.csv")
features = pd.read_csv(f"{DATA_DIR}/features.csv", parse_dates=["Date"])

print("train:", train_raw.shape, "| test:", test_raw.shape,
      "| stores:", stores.shape, "| features:", features.shape)
train_raw.head()


In [ ]:
def merge_sources(df, stores, features):
    # features.csv-ს აქვს დუბლირებული IsHoliday სვეტი, ამოვიღოთ merge-ის დროს კონფლიქტი
    feat = features.drop(columns=["IsHoliday"])
    out = df.merge(stores, on="Store", how="left")
    out = out.merge(feat, on=["Store", "Date"], how="left")
    return out

train_m = merge_sources(train_raw, stores, features)
test_m = merge_sources(test_raw, stores, features)
print(train_m.shape, test_m.shape)


In [ ]:
def clean_data(df, is_train=True):
    df = df.copy()
    n_before = len(df)

    # 1) დუბლიკატების მოცილება
    df = df.drop_duplicates()

    # 2) MarkDown1-5 -- NaN ნიშნავს რომ იმ კვირას პრომოუშენი არ ყოფილა -> 0
    markdown_cols = [c for c in df.columns if c.startswith("MarkDown")]
    df[markdown_cols] = df[markdown_cols].fillna(0)
    df[markdown_cols] = df[markdown_cols].clip(lower=0)  # ცნობილია, რომ ზოგან უარყოფითი მნიშვნელობებია

    # 3) CPI / Unemployment -- test პერიოდში ნაწილობრივ არ არსებობს (features.csv 2013 აგვისტომდეა);
    #    ამოვავსოთ Store-ის მიხედვით ბოლო ცნობილი მნიშვნელობით (forward-fill დროის მიხედვით)
    df = df.sort_values(["Store", "Date"])
    for col in ["CPI", "Unemployment"]:
        df[col] = df.groupby("Store")[col].ffill().bfill()

    # 4) Weekly_Sales -- მხოლოდ train-ში; უარყოფითი გაყიდვები (დაბრუნებები/შეცდომები) ვჭრით 0-ზე
    if is_train:
        neg_count = (df["Weekly_Sales"] < 0).sum()
        df["Weekly_Sales"] = df["Weekly_Sales"].clip(lower=0)
    else:
        neg_count = 0

    # 5) ტიპების გასწორება
    df["IsHoliday"] = df["IsHoliday"].astype(bool)
    df["Store"] = df["Store"].astype(int)
    df["Dept"] = df["Dept"].astype(int)

    n_after = len(df)
    stats = {"rows_before": n_before, "rows_after": n_after,
             "duplicates_removed": n_before - n_after,
             "negative_sales_clipped": int(neg_count),
             "markdown_na_filled": int(df[markdown_cols].isna().sum().sum())}
    return df, stats

train_clean, train_clean_stats = clean_data(train_m, is_train=True)
test_clean, test_clean_stats = clean_data(test_m, is_train=False)

print("TRAIN cleaning stats:", train_clean_stats)
print("TEST cleaning stats:", test_clean_stats)


In [ ]:
with mlflow.start_run(run_name="NBEATS_Cleaning"):
    mlflow.log_params({f"train_{k}": v for k, v in train_clean_stats.items()})
    mlflow.log_params({f"test_{k}": v for k, v in test_clean_stats.items()})
    mlflow.log_metric("train_missing_after_clean", int(train_clean.isna().sum().sum()))
    mlflow.log_metric("test_missing_after_clean", int(test_clean.isna().sum().sum()))
    train_clean.to_csv("train_clean.csv", index=False)
    mlflow.log_artifact("train_clean.csv")
print("Cleaning run logged to MLflow.")


## 3. Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
    df["DayOfYear"] = df["Date"].dt.dayofyear

    # Kaggle-ის ოფიციალური "special" holiday კვირები (WMAE-ში x5 წონა აქვთ)
    superbowl = pd.to_datetime(["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"])
    labor_day = pd.to_datetime(["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"])
    thanksgiving = pd.to_datetime(["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"])
    christmas = pd.to_datetime(["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"])

    df["IsSuperBowl"] = df["Date"].isin(superbowl)
    df["IsLaborDay"] = df["Date"].isin(labor_day)
    df["IsThanksgiving"] = df["Date"].isin(thanksgiving)
    df["IsChristmas"] = df["Date"].isin(christmas)

    # unique_id -- ერთი დროითი მწკრივი თითო Store-Dept წყვილზე (neuralforecast ფორმატი)
    df["unique_id"] = df["Store"].astype(str) + "_" + df["Dept"].astype(str)
    df["ds"] = df["Date"]
    return df

train_fe = engineer_features(train_clean)
test_fe = engineer_features(test_clean)

if "Weekly_Sales" in train_fe.columns:
    train_fe["y"] = train_fe["Weekly_Sales"]

print(train_fe.shape, test_fe.shape)
train_fe[["unique_id", "ds", "y", "IsHoliday", "IsSuperBowl", "IsThanksgiving"]].head()


In [ ]:
feature_cols_created = ["Year", "Month", "WeekOfYear", "DayOfYear",
                        "IsSuperBowl", "IsLaborDay", "IsThanksgiving", "IsChristmas",
                        "unique_id", "ds", "y"]

with mlflow.start_run(run_name="NBEATS_FeatureEngineering"):
    mlflow.log_param("n_features_created", len(feature_cols_created))
    mlflow.log_param("feature_list", ",".join(feature_cols_created))
    mlflow.log_metric("n_unique_series", train_fe["unique_id"].nunique())
    mlflow.log_metric("n_rows_train", len(train_fe))
print("Feature engineering run logged. Unique series:", train_fe["unique_id"].nunique())


## 4. Feature Selection

N-BEATS-ის base ვერსია მხოლოდ target-ის ისტორიას იყენებს, თუმცა `neuralforecast`-ის NBEATS
იმპლემენტაცია იძლევა `hist_exog_list` / `futr_exog_list` პარამეტრებს (NBEATSx-ის ლოგიკა).
ამ სექციაში ვირჩევთ, რომელი გარე ცვლადები ღირს დამატება — corr(target)-ის და missing-rate-ის მიხედვით.

In [ ]:
candidate_exog = ["Temperature", "Fuel_Price", "CPI", "Unemployment",
                   "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]

corr_with_target = train_fe[candidate_exog + ["y"]].corr()["y"].drop("y").sort_values(key=abs, ascending=False)
missing_rate = train_fe[candidate_exog].isna().mean()

selection_table = pd.DataFrame({"abs_corr_with_y": corr_with_target.abs(),
                                 "missing_rate": missing_rate}).sort_values("abs_corr_with_y", ascending=False)
selection_table


In [ ]:
# შერჩევის წესი: |corr| >= 0.02 და missing_rate < 0.5 (ორივე პირობა უნდა სრულდებოდეს)
CORR_THRESHOLD = 0.02
MISSING_THRESHOLD = 0.5

selected_exog = selection_table[(selection_table["abs_corr_with_y"] >= CORR_THRESHOLD) &
                                 (selection_table["missing_rate"] < MISSING_THRESHOLD)].index.tolist()

# IsHoliday-ჯგუფის ცვლადები (მომავალშიც ცნობილია -> futr_exog კანდიდატები)
FUTR_EXOG = ["IsHoliday", "IsSuperBowl", "IsLaborDay", "IsThanksgiving", "IsChristmas"]
# დანარჩენი რიცხვითი მაჩვენებლები ცნობილი არაა მომავალში -> hist_exog (მხოლოდ ისტორიაზე)
HIST_EXOG = [c for c in selected_exog if c not in FUTR_EXOG]

print("Selected futr_exog:", FUTR_EXOG)
print("Selected hist_exog:", HIST_EXOG)


In [ ]:
with mlflow.start_run(run_name="NBEATS_FeatureSelection"):
    mlflow.log_param("corr_threshold", CORR_THRESHOLD)
    mlflow.log_param("missing_threshold", MISSING_THRESHOLD)
    mlflow.log_param("futr_exog", ",".join(FUTR_EXOG))
    mlflow.log_param("hist_exog", ",".join(HIST_EXOG))
    selection_table.to_csv("feature_selection_table.csv")
    mlflow.log_artifact("feature_selection_table.csv")
print("Feature selection run logged.")


## 5. Train / Validation Split (time-based holdout)

Test set-ი მომავალშია (2012-11-02 -> 2013-07-26, 39 კვირა), ამიტომ validation-ისთვისაც
ბოლო **39 კვირას** ვჭრით holdout-ად — ეს იძლევა Kaggle-ის leaderboard-ის რეალურ იმიტაციას.

In [ ]:
max_date = train_fe["ds"].max()
cutoff_date = max_date - pd.Timedelta(weeks=HORIZON)

nf_cols = ["unique_id", "ds", "y"] + HIST_EXOG + FUTR_EXOG
nf_df = train_fe[nf_cols].dropna(subset=["y"]).sort_values(["unique_id", "ds"]).reset_index(drop=True)
# hist/futr exog-ში დარჩენილი NA-ები ვავსებთ 0-ით (მოკლე სერიების კიდეებზე)
nf_df[HIST_EXOG + FUTR_EXOG] = nf_df[HIST_EXOG + FUTR_EXOG].fillna(0)

Y_train_full = nf_df[nf_df["ds"] <= cutoff_date].copy()
Y_valid = nf_df[nf_df["ds"] > cutoff_date].copy()

# მხოლოდ ისეთი სერიები, რომლებსაც აქვთ საკმარისი ისტორია (>= 2*HORIZON) და validation მონაცემები
valid_ids = set(Y_valid["unique_id"]) & set(Y_train_full["unique_id"])
series_counts = Y_train_full.groupby("unique_id").size()
valid_ids = [uid for uid in valid_ids if series_counts.get(uid, 0) >= 2 * HORIZON]

Y_train_full = Y_train_full[Y_train_full["unique_id"].isin(valid_ids)].reset_index(drop=True)
Y_valid = Y_valid[Y_valid["unique_id"].isin(valid_ids)].reset_index(drop=True)

print(f"Cutoff date: {cutoff_date.date()}  |  usable series: {len(valid_ids)}")
print("Train rows:", len(Y_train_full), "| Valid rows:", len(Y_valid))


In [ ]:
# hyperparameter search-ისთვის ვირჩევთ სემფლს, რომ თითო run 25 წუთს არ სცდებოდეს
rng = np.random.RandomState(RANDOM_SEED)
sample_ids = rng.choice(sorted(valid_ids), size=min(HP_SEARCH_SAMPLE_N, len(valid_ids)), replace=False)

Y_train_sample = Y_train_full[Y_train_full["unique_id"].isin(sample_ids)].reset_index(drop=True)
Y_valid_sample = Y_valid[Y_valid["unique_id"].isin(sample_ids)].reset_index(drop=True)
print("HP-search series:", len(sample_ids), "| train rows:", len(Y_train_sample))


## 6. WMAE — Kaggle-ის ოფიციალური შეფასების მეტრიკა

$$WMAE = \dfrac{\sum_i w_i |y_i - \hat y_i|}{\sum_i w_i}, \quad w_i = 5 \text{ თუ holiday კვირაა, სხვა შემთხვევაში } 1$$

In [ ]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(np.asarray(is_holiday).astype(bool), 5, 1)
    return float(np.sum(weights * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(weights))


## 7. Hyperparameter Search

**პარამეტრები, რომლებსაც ვცდით:**
- `input_size` — რამდენი წარსული კვირა ეძლევა კონტექსტად (context length)
- `stack_types` / `n_blocks` — ინტერპრეტირებადი (trend + seasonality) vs generic არქიტექტურა
- `mlp_units` — თითო ბლოკის fully-connected ფენების სიგანე
- `learning_rate`
- `batch_size` / `windows_batch_size`

თითოეული კონფიგურაცია ვარჯიშდება `Y_train_sample`-ზე (300 სემფლირებული სერია),
`trainer_kwargs={"max_time": {"minutes": MAX_RUN_MINUTES}}`-ით შემოსაზღვრული, რაც
გარანტიას იძლევა, რომ ვერცერთი run 30 წუთს არ გადააჭარბებს.

In [ ]:
HP_GRID = [
    {"name": "interpretable_short_ctx",
     "input_size": 2 * HORIZON, "stack_types": ["trend", "seasonality"],
     "n_blocks": [3, 3], "mlp_units": [[256, 256]] * 2,
     "n_harmonics": 2, "n_polynomials": 2,
     "learning_rate": 1e-3, "batch_size": 32, "windows_batch_size": 256},

    {"name": "interpretable_long_ctx",
     "input_size": 3 * HORIZON, "stack_types": ["trend", "seasonality"],
     "n_blocks": [3, 3], "mlp_units": [[256, 256]] * 2,
     "n_harmonics": 2, "n_polynomials": 2,
     "learning_rate": 1e-3, "batch_size": 32, "windows_batch_size": 256},

    {"name": "generic_wide",
     "input_size": 2 * HORIZON, "stack_types": ["identity", "identity"],
     "n_blocks": [3, 3], "mlp_units": [[512, 512]] * 2,
     "n_harmonics": 2, "n_polynomials": 2,
     "learning_rate": 1e-3, "batch_size": 32, "windows_batch_size": 256},

    {"name": "generic_deep_lowlr",
     "input_size": 2 * HORIZON, "stack_types": ["identity", "identity"],
     "n_blocks": [4, 4], "mlp_units": [[256, 256]] * 2,
     "n_harmonics": 2, "n_polynomials": 2,
     "learning_rate": 5e-4, "batch_size": 32, "windows_batch_size": 256},

    {"name": "interpretable_small_batch",
     "input_size": 2 * HORIZON, "stack_types": ["trend", "seasonality"],
     "n_blocks": [3, 3], "mlp_units": [[256, 256]] * 2,
     "n_harmonics": 2, "n_polynomials": 2,
     "learning_rate": 1e-3, "batch_size": 16, "windows_batch_size": 128},

    {"name": "generic_highlr",
     "input_size": 2 * HORIZON, "stack_types": ["identity", "identity"],
     "n_blocks": [3, 3], "mlp_units": [[256, 256]] * 2,
     "n_harmonics": 2, "n_polynomials": 2,
     "learning_rate": 3e-3, "batch_size": 32, "windows_batch_size": 256},
]
print(f"{len(HP_GRID)} configurations to test, each capped at {MAX_RUN_MINUTES} minutes.")


In [ ]:
def build_nbeats(cfg, hist_exog, futr_exog, max_minutes):
    return NBEATS(
        h=HORIZON,
        input_size=cfg["input_size"],
        stack_types=cfg["stack_types"],
        n_blocks=cfg["n_blocks"],
        mlp_units=cfg["mlp_units"],
        n_harmonics=cfg.get("n_harmonics", 2),
        n_polynomials=cfg.get("n_polynomials", 2),
        hist_exog_list=hist_exog,
        futr_exog_list=futr_exog,
        loss=MAE(),
        learning_rate=cfg["learning_rate"],
        batch_size=cfg["batch_size"],
        windows_batch_size=cfg["windows_batch_size"],
        max_steps=100_000,          # ფაქტობრივ ლიმიტს akეთებს max_time (ქვემოთ)
        early_stop_patience_steps=5,
        val_check_steps=50,
        scaler_type="standard",
        random_seed=RANDOM_SEED,
        trainer_kwargs={"max_time": {"minutes": max_minutes}, "enable_progress_bar": False},
    )


In [ ]:
def evaluate_on_valid(nf, valid_df, train_df):
    futr_cols = ["unique_id", "ds"] + FUTR_EXOG
    futr_df = valid_df[futr_cols].copy()
    preds = nf.predict(futr_df=futr_df)
    merged = preds.merge(valid_df[["unique_id", "ds", "y"]], on=["unique_id", "ds"], how="inner")
    is_holiday = valid_df.set_index(["unique_id", "ds"]).loc[
        list(zip(merged["unique_id"], merged["ds"])), "IsHoliday"
    ].values if "IsHoliday" in valid_df.columns else np.zeros(len(merged))
    model_col = [c for c in preds.columns if c not in ("unique_id", "ds")][0]
    score = wmae(merged["y"], merged[model_col], is_holiday)
    mae = float(np.mean(np.abs(merged["y"] - merged[model_col])))
    return score, mae, model_col


In [ ]:
results = []

for cfg in HP_GRID:
    run_name = f"NBEATS_HPSearch_{cfg['name']}"
    print(f"\n=== {run_name} ===")
    t0 = time.time()
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({k: (str(v) if isinstance(v, (list, dict)) else v)
                            for k, v in cfg.items() if k != "name"})
        mlflow.log_param("max_time_minutes", MAX_RUN_MINUTES)
        mlflow.log_param("n_series_used", len(sample_ids))

        model = build_nbeats(cfg, HIST_EXOG, FUTR_EXOG, MAX_RUN_MINUTES)
        nf = NeuralForecast(models=[model], freq="W-FRI")
        nf.fit(df=Y_train_sample)

        elapsed_min = (time.time() - t0) / 60
        try:
            wmae_score, mae_score, model_col = evaluate_on_valid(nf, Y_valid_sample, Y_train_sample)
        except Exception as e:
            print("Eval failed:", e)
            wmae_score, mae_score = np.inf, np.inf

        mlflow.log_metric("val_WMAE", wmae_score)
        mlflow.log_metric("val_MAE", mae_score)
        mlflow.log_metric("train_minutes", elapsed_min)

        results.append({**{k: v for k, v in cfg.items()}, "val_WMAE": wmae_score,
                         "val_MAE": mae_score, "train_minutes": elapsed_min})
        print(f"WMAE={wmae_score:.2f} | MAE={mae_score:.2f} | {elapsed_min:.1f} min")

results_df = pd.DataFrame(results).sort_values("val_WMAE")
results_df


In [ ]:
best_cfg_name = results_df.iloc[0]["name"]
best_cfg = next(c for c in HP_GRID if c["name"] == best_cfg_name)
print("საუკეთესო კონფიგურაცია:", best_cfg_name)
best_cfg


## 8. საბოლოო ვარჯიში სრულ მონაცემებზე

საუკეთესო კონფიგურაციით ვარჯიშდება მოდელი **ყველა** (`valid_ids`) სერიაზე, ისევ `max_time`
ლიმიტით — თუ სრულ დატასეტზე კონვერგენცია ვერ ესწრება, `NeuralForecast`/PyTorch-Lightning
უბრალოდ წყვეტს ვარჯიშს დროის ამოწურვისას (ე.ი. deployment-ისთვის საკმარისად სტაბილურია).

In [ ]:
with mlflow.start_run(run_name="NBEATS_Final_Training") as final_run:
    mlflow.log_params({k: (str(v) if isinstance(v, (list, dict)) else v)
                        for k, v in best_cfg.items() if k != "name"})
    mlflow.log_param("max_time_minutes", MAX_RUN_MINUTES)
    mlflow.log_param("n_series_used", len(valid_ids))
    mlflow.log_param("hist_exog", ",".join(HIST_EXOG))
    mlflow.log_param("futr_exog", ",".join(FUTR_EXOG))

    t0 = time.time()
    final_model = build_nbeats(best_cfg, HIST_EXOG, FUTR_EXOG, MAX_RUN_MINUTES)
    nf_final = NeuralForecast(models=[final_model], freq="W-FRI")
    nf_final.fit(df=Y_train_full)
    elapsed_min = (time.time() - t0) / 60

    final_wmae, final_mae, model_col = evaluate_on_valid(nf_final, Y_valid, Y_train_full)
    mlflow.log_metric("val_WMAE", final_wmae)
    mlflow.log_metric("val_MAE", final_mae)
    mlflow.log_metric("train_minutes", elapsed_min)

    nf_final.save(path="./nbeats_final_model", overwrite=True)
    mlflow.log_artifacts("./nbeats_final_model", artifact_path="nbeats_final_model")

    final_run_id = final_run.info.run_id

print(f"Final WMAE={final_wmae:.2f} | MAE={final_mae:.2f} | {elapsed_min:.1f} min | run_id={final_run_id}")


## 9. Pipeline-ის დარეგისტრირება MLflow Model Registry-ში

`NeuralForecast` ობიექტს ვახვევთ `mlflow.pyfunc.PythonModel`-ში, რომ `model_inference.ipynb`-მა
პირდაპირ `mlflow.pyfunc.load_model` + `.predict()` შეძლოს, ისე რომ raw test set-ზე (preprocess-ის
გარეშე) გაეშვას — ამისთვის wrapper-ი შიგნით იმეორებს cleaning + feature engineering ნაბიჯებს.

In [ ]:
class NBEATSPipeline(mlflow.pyfunc.PythonModel):
    """სრული pipeline: raw dataframe -> cleaning -> feature engineering -> NBEATS პროგნოზი."""

    def load_context(self, context):
        self.nf = NeuralForecast.load(path=context.artifacts["nf_model_dir"])
        self.hist_exog = HIST_EXOG
        self.futr_exog = FUTR_EXOG

    def predict(self, context, model_input: pd.DataFrame):
        df = model_input.copy()
        df["Date"] = pd.to_datetime(df["Date"])
        df, _ = clean_data(df, is_train=False)
        df = engineer_features(df)
        futr_cols = ["unique_id", "ds"] + self.futr_exog
        futr_df = df[futr_cols].fillna(0)
        preds = self.nf.predict(futr_df=futr_df)
        model_col = [c for c in preds.columns if c not in ("unique_id", "ds")][0]
        preds = preds.rename(columns={model_col: "Weekly_Sales_Pred"})
        return preds


In [ ]:
with mlflow.start_run(run_id=final_run_id):
    mlflow.pyfunc.log_model(
        artifact_path="nbeats_pipeline",
        python_model=NBEATSPipeline(),
        artifacts={"nf_model_dir": "./nbeats_final_model"},
        registered_model_name="WalmartSales_NBEATS",
    )
print("Pipeline დარეგისტრირდა როგორც 'WalmartSales_NBEATS' Model Registry-ში.")


## 10. შეჯამება

- ვცადეთ 6 ჰიპერპარამეტრული კონფიგურაცია (interpretable trend+seasonality vs. generic stacks,
  სხვადასხვა `input_size`, `learning_rate`, `batch_size`) — თითოეული შემოსაზღვრული 25 წუთით
  `trainer_kwargs={"max_time": ...}`-ის საშუალებით.
- საუკეთესო კონფიგურაცია საბოლოოდ გადავწვართეთ სრულ დატასეტზე (ყველა Store-Dept სერია).
- საბოლოო pipeline (cleaning + feature engineering + NBEATS მოდელი) დარეგისტრირებულია
  MLflow Model Registry-ში სახელით `WalmartSales_NBEATS` და მზადაა `model_inference.ipynb`-ში
  გამოსაყენებლად test set-ზე submission-ის დასაგენერირებლად.
- შედეგები (`results_df`) და საბოლოო validation WMAE/MAE ჩაწერილია MLflow-ში — README.md-ში
  ეს რიცხვები შედარებული უნდა იყოს დანარჩენი არქიტექტურების (LightGBM/XGBoost, N-BEATS-ის
  გარდა სხვა DL, ARIMA/SARIMA) შედეგებთან.
